# SOCO - DINO-first semantic correspondence pipeline
Stage 1: frozen DINOv2 + nearest-neighbour matching -> Stage 2: fine-tune last blocks -> Stage 3: advanced methods.
Each stage is gated by validation PCK (`decide`). Attach a Kaggle dataset containing this repo (`soco/`, `run.py`)
and, for offline runs, the DINOv2 repo source + checkpoint.

In [ ]:
import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = "/kaggle/input/datasets/langchenneto/soco-baseline-pipeline"          
sys.path.insert(0, REPO)

from pathlib import Path
from soco.config import Config
from soco.evaluate import evaluate, sweep_matching, sweep_features
from soco.decide import recommend
from PIL import Image
from torch.utils.data import Dataset

# The competition dataset:

In [ ]:
class AMLQueryDataset(Dataset):
    """One source query and a set of accepted targets per row."""

    def __init__(self, df, root, max_queries=0):
        self.df = df.reset_index(drop=True).copy()
        if max_queries:
            self.df = self.df.iloc[:max_queries].copy()
        self.root = Path(root)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        src_path = self.root / row["source_image"]
        trg_path = self.root / row["target_image"]

        with Image.open(src_path) as im:
            src_img = im.convert("RGB")
        with Image.open(trg_path) as im:
            trg_img = im.convert("RGB")

        targets = np.asarray(
            json.loads(row["target_points"]), dtype=np.float32
        ).reshape(-1, 2)
        scale = float(row["scale"])

        if len(targets) == 0 or not np.isfinite(targets).all():
            raise ValueError(f"Invalid target_points for {row['id']}")
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"Invalid scale for {row['id']}")

        return {
            "pair_id": str(row["id"]),       # Unique query ID
            "image_pair_id": str(row["pair_id"]),
            "category": "all",
            "src_path": str(src_path),
            "trg_path": str(trg_path),
            "src_img": src_img,
            "trg_img": trg_img,
            "src_kps": np.asarray(
                [[row["source_x"], row["source_y"]]], dtype=np.float32
            ),
            "trg_kps": None,                 # No one-to-one target label
            "trg_bbox": None,
            "accepted_targets": targets,
            "scale": scale,
        }

In [ ]:
# Configuration cell:

root = Path("/kaggle/input/competitions/aml-challenge-2026-27")

cfg = Config(
    data_root = str(root),   
    data_format ="csv",
    csv_path = str(root / "test.csv"),
    image_dir = str(root),
    backbone = "dinov2_vitb14",
    # Offline (internet disabled): point to the attached DINOv2 source + weights
    hub_repo ="/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2", hub_local=True,
    weights_path ="/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2_assets/dinov2_vitb14.pth",
    img_size =518, max_pairs=500, out_dir="/kaggle/working/outputs",
)

# The competition metric:

In [ ]:
import soco.evaluate as eval_module
from soco.metrics import PCKMeter

class AMLPCKMeter(PCKMeter):
    def update(self, sample, pred, flipped=False):
        pred = np.asarray(pred, dtype=np.float64)

        if pred.shape != (1, 2) or not np.isfinite(pred).all():
            raise ValueError(
                f"Expected one finite (x, y) prediction; got {pred}"
            )

        targets = np.asarray(
            sample["accepted_targets"], dtype=np.float64
        )
        distance = np.linalg.norm(targets - pred[0], axis=1).min()
        normalized_error = distance / sample["scale"]

        self.rows.append({
            "pair_id": sample["pair_id"],
            "category": sample["category"],
            "kp": 0,
            "dist": float(normalized_error),
            "flipped": flipped,
        })

    def frame(self):
        df = pd.DataFrame(
            self.rows,
            columns=["pair_id", "category", "kp", "dist", "flipped"],
        )
        for alpha in self.alphas:
            df[f"pck@{alpha}"] = (df["dist"] < alpha).astype(float)
        return df


# evaluate(), sweep_matching(), and sweep_features() use this module global.
eval_module.PCKMeter = AMLPCKMeter

# Held-out validation dataset (new build)

In [ ]:
train = pd.read_csv(root / "train.csv")

pair_ids = train["pair_id"].drop_duplicates().to_numpy()
assert len(pair_ids) >= 2

rng = np.random.default_rng(42)
rng.shuffle(pair_ids)

n_val = min(len(pair_ids) - 1, max(1, int(0.2 * len(pair_ids))))
val_ids = set(pair_ids[:n_val])

is_val = train["pair_id"].isin(val_ids)
train_df = train.loc[~is_val].copy()
val_df = train.loc[is_val].sample(frac=1, random_state=42)

assert set(train_df["pair_id"]).isdisjoint(val_df["pair_id"])

val_ds = AMLQueryDataset(val_df, root, max_queries=500)

sample = val_ds[0]
print("Validation queries:", len(val_ds))
print("Source point:", sample["src_kps"])
print("Accepted targets:", sample["accepted_targets"])
print("Scale:", sample["scale"])

## Stage 1a - matching sweep (features computed once)

In [ ]:
assert len(val_ds) > 0
assert eval_module.PCKMeter is AMLPCKMeter

sweep = eval_module.sweep_matching(
    cfg.replace(run_name="s1"),
    {
        "matcher": ["argmax", "soft_window"],
        "feat_upsample": [1, 2],
        "soft_temperature": [0.02, 0.04],
    },
    ds=val_ds,
)

best = sweep.iloc[0][
    ["matcher", "feat_upsample", "soft_temperature"]
].to_dict()

best["feat_upsample"] = int(best["feat_upsample"])

print("Best settings:", best)
print("Validation PCK:", sweep.iloc[0]["pck@0.1 (per-kp)"] / 100)

## Stage 1b - feature sweep (layer, resolution)

In [ ]:
feat = eval_module.sweep_features(
    cfg.replace(run_name="s1f", **best),
    {
        "layer": [-1, -2],
        "img_size": [448, 518, 644],
    },
    ds=val_ds,
)

display(feat.head())

## Stage 1c - evaluate best config and decide

In [ ]:
top = feat.iloc[0]

cfg1 = cfg.replace(
    run_name="stage1_best",
    layer=int(top["layer"]),
    img_size=int(top["img_size"]),
    **best,
)

summ, meter = eval_module.evaluate(cfg1, ds=val_ds)

print("Validation PCK:", summ["pck@0.1 (per-kp)"] / 100)
print(recommend(f"{cfg1.out_dir}/stage1_best", target=70.0))

## Visual check (do this before trusting any number)

In [ ]:
from soco.evaluate import load_model
from soco.matching import predict_pair

model, dev = load_model(cfg1)
s = val_ds[0]

pred, _ = predict_pair(model, s, cfg1, dev)
targets = s["accepted_targets"]

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(s["src_img"])
ax[0].scatter(*s["src_kps"].T, c="red", s=60)
ax[0].set_title("Source query")

ax[1].imshow(s["trg_img"])
ax[1].scatter(*targets.T, c="lime", s=60, label="Accepted targets")
ax[1].scatter(*pred.T, c="red", marker="x", s=100, label="Prediction")
ax[1].set_title("Target prediction")
ax[1].legend()

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

## Stage 2 - fine-tune (only if `decide` says so). Needs GPU.

In [ ]:
from soco.finetune import finetune
cfg2 = cfg1.replace(run_name="stage2_ft2", ft_unfreeze_blocks=2, ft_epochs=2, max_pairs=500)
best_val = finetune(cfg2)
print(recommend(f"{cfg2.out_dir}/stage2_ft2/val_ep{cfg2.ft_epochs-1}", target=70.0,
                baseline_dir=f"{cfg1.out_dir}/stage1_best", stage=2))

## Submission

In [ ]:
raw = pd.read_csv(root / "test.csv")

adapted = pd.DataFrame({
    "pair_id": raw["id"],
    "src_image": raw["source_image"],
    "trg_image": raw["target_image"],
    "src_x": raw["source_x"],
    "src_y": raw["source_y"],
})

adapted_path = Path("/kaggle/working/test_adapted.csv")
adapted.to_csv(adapted_path, index=False)

In [ ]:
from soco.submit import predict_split

final = cfg1

test_cfg = final.replace(
    data_format="csv",
    max_pairs=0,
    csv_path=str(adapted_path),
    image_dir=str(root),
)

submission = predict_split(
    test_cfg,
    split="test",
    out_csv="/kaggle/working/submission.csv",
    sample_submission=str(root / "sample_submission.csv"),
)

assert len(submission) == len(raw)
assert submission["id"].is_unique